## Phase 3, Step 1: Generating the Final Forecast

In [0]:
# 1. Install LightGBM using the magic command
%pip install lightgbm

In [0]:
import pandas as pd
import joblib
import gc

# 1. Load the "Gold" Requirements
# We only need the test set and the trained model file
test_path = '/Volumes/workspace/default/m5_walmart_data/m5_test.parquet'
model_path = '/Volumes/workspace/default/m5_walmart_data/m5_forecasting_model.pkl'

print("Loading Test Data and Model Artifact...")
test_df = pd.read_parquet(test_path)
model = joblib.load(model_path)

# 2. Define Features used in training
unused = ['id', 'sales', 'wm_yr_wk', 'day', 'day_int']
features = [c for c in test_df.columns if c not in unused]

X_test = test_df[features]
y_test = test_df['sales']

print(f"Ready for Inference on {len(X_test)} rows.")

## Phase 3, Step 2: Generating the Gold Forecast

In [0]:
import numpy as np

# 1. Generate Predictions
print("Calculating final forecasts...")
test_df['predicted_sales'] = model.predict(X_test)

# 2. Business Logic: Clip negative values
# In retail, you cannot sell -5 items; we round up to 0 for realism.
test_df['predicted_sales'] = test_df['predicted_sales'].clip(lower=0)

# 3. Create a clean 'Gold' view
# We keep only the IDs, the actuals, and our predictions for reporting.
gold_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 
             'day', 'sales', 'predicted_sales']

gold_output = test_df[gold_cols].copy()
gold_output.rename(columns={'sales': 'actual_sales'}, inplace=True)

print("Inference Complete.")
gold_output.head()

## Phase 3, Step 3: Final Persistence & Error Analysis

In [0]:
# 1. Calculate Forecast Bias
# Bias = (Forecast - Actual) / Actual. It tells us if we are overstocking.
total_actual = gold_output['actual_sales'].sum()
total_pred = gold_output['predicted_sales'].sum()
bias = (total_pred - total_actual) / total_actual

print(f"Total Actual Sales: {total_actual:,.0f}")
print(f"Total Predicted Sales: {total_pred:,.0f}")
print(f"Overall Forecast Bias: {bias:.2%}")

# 2. Save the Gold Table
gold_path = '/Volumes/workspace/default/m5_walmart_data/m5_gold_final_forecast.parquet'
gold_output.to_parquet(gold_path, index=False)

print(f"Gold Layer saved successfully: {gold_path}")

In [0]:
# 1. Create a smaller sample for the web dashboard 
# We take the first 500 unique product IDs to keep the file size small
dash_sample = gold_output[gold_output['id'].isin(gold_output['id'].unique()[:500])]

# 2. Save it as a CSV
dash_sample.to_csv('/tmp/m5_dashboard_sample.csv', index=False)

# 3. Move it to your Volume so you can download it easily
import shutil
shutil.copyfile('/tmp/m5_dashboard_sample.csv', '/Volumes/workspace/default/m5_walmart_data/m5_dashboard_sample.csv')

print("Dashboard sample created in your Volume!")

In [0]:
# 1. Get feature importance from the LightGBM model
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False).head(10)

# 2. Save to CSV for the dashboard
importance_df.to_csv('/tmp/feature_importance.csv', index=False)

# 3. Move to Volume for download
import shutil
shutil.copyfile('/tmp/feature_importance.csv', '/Volumes/workspace/default/m5_walmart_data/feature_importance.csv')

print("Feature importance data ready for download!")